# 08 · Reranking: drugi etap oceny kandydatów

Retriever szybko zbiera szeroką pulę. Reranker czyta **pytanie i tekst kandydata razem**, ocenia czy fragment naprawdę odpowiada i porządkuje krótką listę. Poznamy cross-encoder, późną interakcję, generatywny LLM i punktowy model decyzyjny. Uruchomimy lokalny **System One Clef Flash i Clef** przez Ollama. To modele decyzyjne; nie wywołuj ich przez `chat`.

Wynik Clef jest osądem relewancji, nie prawdopodobieństwem prawdy. Weryfikacja faktu nadal wymaga oryginalnego fragmentu i jego metadanych.

In [ ]:
# Jedyna komórka z importami i ścieżką projektu.
import os
import re
import sys
import time
from pathlib import Path

import numpy as np
from dotenv import dotenv_values
from ollama import Client
from rank_bm25 import BM25Okapi

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import embed_texts, ollama_settings, section_records

## 1. Kandydaci z dwóch tanich wyszukiwarek

Budujemy małe RRF jak w lekcji 07, ale notebook działa samodzielnie. Reranker dostaje tylko top-6; koszt rośnie z liczbą i długością fragmentów. Długie wejście może być obcięte przez model, dlatego podajemy jawnie ograniczony tekst i nie traktujemy oceny jako dowodu dla niewidzianej reszty.

In [ ]:
records = section_records()
question = "Why is a percentage forecast error undefined when demand is zero?"
bm25 = BM25Okapi([re.findall("[a-z0-9]+", row["text"].lower()) for row in records])
lexical = bm25.get_scores(re.findall("[a-z0-9]+", question.lower()))
vectors = embed_texts([row["text"] for row in records])
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
q = embed_texts([question])[0]
q /= np.linalg.norm(q)
dense = vectors @ q
fused = np.zeros(len(records))
for ranking in (np.argsort(-lexical)[:10], np.argsort(-dense)[:10]):
    for rank, index in enumerate(ranking, 1):
        fused[index] += 1 / (60 + rank)
candidate_indices = np.argsort(-fused)[:6]
candidates = [records[int(i)] for i in candidate_indices]
[(row["id"], row["heading"]) for row in candidates]

## 2. System One: jedno pytanie typu score na kandydata

`state` zawiera pytanie i jawnie ponumerowane teksty. Dla każdego ID przekazujemy kryteria 0–3: nieistotny, temat pokrewny, częściowa odpowiedź, bezpośrednia odpowiedź. API zwraca rozkład ocen i **ważoną ocenę**; sortujemy ją malejąco. Nie nazywamy jej „confidence, że fragment jest prawdziwy”.

Clef Flash jest szybszym wariantem; Clef większym. Uruchomienia są niezależne, a nazwy można ustawić przez `RAG_RERANK_MODELS` w root `.env` (lista po przecinkach). Jeśli lokalne tagi są stare, `ollama pull clef-flash:latest` i `ollama pull clef:latest` odświeży format System One.

In [ ]:
def rerank_with_systemone(model, question, candidates):
    state = {"query": question, "passages": [{"id": row["id"], "text": row["text"][:1100]} for row in candidates]}
    questions = {
        f"p{i}": {"type": "score", "instructions": f"Rate only passage {i} for whether it directly answers the query; ignore other passages.",
                 "criteria": ["Unrelated", "Same broad topic but no answer", "Partly answers", "Directly answers with relevant evidence"]}
        for i in range(len(candidates))
    }
    response = Client(host=ollama_settings()[0]).systemone(model=model, state=state, questions=questions)
    return sorted([{"id": row["id"], "score_0_to_3": float(response.answers[f"p{i}"].score)}
                   for i, row in enumerate(candidates)], key=lambda row: -row["score_0_to_3"])


config = dotenv_values(ROOT / ".env")
model_names = (os.environ.get("RAG_RERANK_MODELS") or config.get("RAG_RERANK_MODELS") or "clef-flash:latest,clef:latest").split(",")
comparisons = {}
for model in model_names:
    start = time.perf_counter()
    comparisons[model.strip()] = {"ranking": rerank_with_systemone(model.strip(), question, candidates),
                                  "seconds": round(time.perf_counter() - start, 2)}
comparisons

## 3. Odczytaj różnicę, nie tylko nową listę

Reranker może przesunąć właściwy fragment z miejsca 4 na 1, ale może też popełnić błąd. Porównaj top-1 obu modeli z tekstem źródła. Gdy wyniki są rozbieżne, to sygnał do inspekcji, nie do automatycznego głosowania. Oceniaj `nDCG@k`, recall kandydatów, latency i koszt na ręcznie oznaczonych pytaniach.

In [ ]:
original = [row["id"] for row in candidates]
{"retriever_top3": original[:3],
 "reranked_top3": {model: [item["id"] for item in data["ranking"][:3]] for model, data in comparisons.items()},
 "first_source_excerpt": next(row["text"][:350] for row in records if row["id"] == next(iter(comparisons.values()))["ranking"][0]["id"])}

### Typy rerankerów

| Typ | Co widzi | Plus | Koszt / ograniczenie |
|---|---|---|---|
| Cross-encoder | query i jeden fragment naraz | precyzyjna ocena pary | koszt dla każdego kandydata |
| Późna interakcja (np. ColBERT) | tokenowe reprezentacje obu stron | dokładniejsze dopasowanie tokenów | większy indeks i złożoność |
| Generatywny LLM | listę lub pary w promptcie | można poprosić o uzasadnienie | czas, halucynacje, niestabilny format |
| Clef System One | tekst stanu i jawne kryteria score | osądy wielu fragmentów jednym wywołaniem | ocena zależy od kryteriów i limitu wejścia |

**Ćwiczenie:** zmień pytanie na takie, którego korpus nie obsługuje. Czy nawet najwyższa ocena oznacza wystarczający dowód? Dodaj osobny warunek odrzucenia lub ocenę wystarczalności. [Oficjalna karta Clef](https://ollama.com/library/clef) opisuje API `systemone`, typ `score` i znaczenie `confidence`.